# TrustOCT — Training Notebook 1: Baseline Models
### Models: `resnet50` + `msf_resnet50`


In [ ]:
import os
if not os.path.exists('src'):
    os.system('git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git repo_temp')
    os.system('cp -r repo_temp/* . && rm -rf repo_temp')
try:
    import albumentations, ptflops
except ImportError:
    os.system('pip install -r requirements.txt')


In [ ]:
import os, sys, time, cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f'Drive mount note: {e}')


In [ ]:
# Download Kermany OCT2017 dataset from Kaggle
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print('Please upload kaggle.json API token:')
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            os.system('mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json')
            os.system('kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany')
    except Exception as e:
        print(f'Dataset note: {e}')


In [ ]:
from src.datasets.dataset import auto_detect_columns, patient_level_split
shared_folder = '/content/drive/MyDrive/TrustOCT_Results'
os.makedirs(shared_folder, exist_ok=True)
os.makedirs(os.path.join(shared_folder, 'outputs'), exist_ok=True)

csv_path = 'kermany_dataset_mapping.csv'
if not os.path.exists(csv_path) and os.path.exists(os.path.join(shared_folder, 'kermany_dataset_mapping.csv')):
    import shutil
    shutil.copy(os.path.join(shared_folder, 'kermany_dataset_mapping.csv'), csv_path)

if os.path.exists(csv_path):
    df = auto_detect_columns(pd.read_csv(csv_path))
    train_df, val_df, test_df = patient_level_split(df)
    print(f'Dataset Split -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')


In [ ]:
epochs = 30
lr = 1e-4
batch_size = 32


### 1. Train `resnet50` Baseline Model


In [ ]:
from src.training.trainer import run_experiment
run_experiment('resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))


### 2. Train `msf_resnet50` Model


In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))


In [ ]:
import shutil
shared_outputs = '/content/drive/MyDrive/TrustOCT_Results/outputs'
os.makedirs(shared_outputs, exist_ok=True)
for m in ['resnet50', 'msf_resnet50']:
    if os.path.exists(f'outputs/{m}'):
        dst = os.path.join(shared_outputs, m)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(f'outputs/{m}', dst)
        print(f'Synced {m} results to Drive.')
